# Continual learning with Split MNIST: a plain SGD baseline

We will train **one model, sequentially**, on five tasks: digits `(0, 1)`, `(2, 3)`, `(4, 5)`, `(6, 7)`, and `(8, 9)`. Each task supplies only its two digits during training. After each task, we test the same model on every task learned so far. A drop in older task accuracy is **catastrophic forgetting**.

This is a deliberately simple baseline: a sizable multilayer perceptron, a single 10-class output layer, cross-entropy loss, and ordinary mini-batch SGD. There is no replay, regularization, task-specific head, or access to the task ID at test time. The notebook stops after measuring this baseline; later methods can be added in a separate experiment.

**Run order:** install dependencies, run the data and model cells, then run the CUDA-only experiment and plots. The first run downloads MNIST into `data/`. Results depend on the seed, hardware, and installed package versions; the charts below are generated from your run rather than filled with example numbers.


## 1. Environment and reproducibility

From this project directory, run `uv sync` and open this notebook with the project's `.venv` kernel. If you use a separate CUDA-enabled PyTorch installation, ensure this kernel points to that environment. The experiment cell explicitly skips training when `torch.cuda.is_available()` is false; data inspection can still run on a CPU.

We fix random seeds for a repeatable task order and weight initialization. Some CUDA operations may still vary slightly between runs.


In [1]:
from pathlib import Path
import random

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda") if torch.cuda.is_available() else None
print(f"PyTorch {torch.__version__} | CUDA available: {DEVICE is not None}")
if DEVICE is not None:
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("Data exploration is available, but the experiment will skip training.")


ModuleNotFoundError: No module named 'matplotlib'

## 2. Look at the data before splitting it

MNIST has 60,000 training images and 10,000 test images. Each is a 28 × 28 grayscale digit labeled from 0 to 9. We keep the original digit labels, rather than relabeling every pair to 0 and 1, because the model must eventually distinguish all ten digits with one output layer.

`ToTensor()` scales pixels to `[0, 1]`. No extra augmentation or normalization is needed for this baseline.


In [ ]:
DATA_DIR = Path("data")
transform = transforms.ToTensor()
train_dataset = datasets.MNIST(DATA_DIR, train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(DATA_DIR, train=False, download=True, transform=transform)

print(f"Train: {len(train_dataset):,} images | Test: {len(test_dataset):,} images")
print(f"Image shape: {tuple(train_dataset[0][0].shape)} | Pixel range: [0, 1]")
print("Training count per digit:", torch.bincount(train_dataset.targets, minlength=10).tolist())
print("Test count per digit:    ", torch.bincount(test_dataset.targets, minlength=10).tolist())


In [ ]:
# One representative training image per digit. These raw examples are not model predictions.
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for digit, ax in enumerate(axes.flat):
    index = torch.nonzero(train_dataset.targets == digit, as_tuple=True)[0][0].item()
    image, label = train_dataset[index]
    ax.imshow(image.squeeze(0), cmap="gray", vmin=0, vmax=1)
    ax.set_title(f"Label {label}")
    ax.axis("off")
fig.suptitle("A sample from each MNIST class")
fig.tight_layout()
plt.show()


In [ ]:
digits = np.arange(10)
train_counts = torch.bincount(train_dataset.targets, minlength=10).numpy()
test_counts = torch.bincount(test_dataset.targets, minlength=10).numpy()
fig, ax = plt.subplots(figsize=(9, 3.5))
ax.bar(digits - 0.2, train_counts, width=0.4, label="Train")
ax.bar(digits + 0.2, test_counts, width=0.4, label="Test")
ax.set(xlabel="Digit", ylabel="Number of images", title="Class counts before task splitting", xticks=digits)
ax.legend()
fig.tight_layout()
plt.show()


## 3. Define the continual-learning stream

The five tasks arrive in the fixed order below. We create disjoint subsets from the official training split and matching subsets from the official test split. A training loader sees only the current pair. Test loaders never contribute gradients.

At evaluation, the model chooses among **all ten outputs**, with no masking and no task hint. This is a class-incremental setting. Chance accuracy for an untrained 10-way classifier is roughly 10%; an old task can even fall below that if the model strongly prefers recent digits. Before all classes have been learned, accuracy on unseen tasks is not a useful forgetting measure, so later plots focus on tasks already introduced.


In [ ]:
TASKS = [(0, 1), (2, 3), (4, 5), (6, 7), (8, 9)]
TASK_NAMES = [f"{a}/{b}" for a, b in TASKS]
BATCH_SIZE = 128


def subset_for_digits(dataset, pair):
    # targets are stored separately, so selecting indices does not decode every image.
    mask = (dataset.targets == pair[0]) | (dataset.targets == pair[1])
    indices = torch.nonzero(mask, as_tuple=True)[0].tolist()
    return Subset(dataset, indices)


train_subsets = [subset_for_digits(train_dataset, pair) for pair in TASKS]
test_subsets = [subset_for_digits(test_dataset, pair) for pair in TASKS]
assert sum(map(len, train_subsets)) == len(train_dataset)
assert sum(map(len, test_subsets)) == len(test_dataset)

# num_workers=0 keeps the notebook portable; pinning helps host-to-GPU copies.
loader_options = {"batch_size": BATCH_SIZE, "num_workers": 0, "pin_memory": DEVICE is not None}
train_loaders = [DataLoader(ds, shuffle=True, **loader_options) for ds in train_subsets]
test_loaders = [DataLoader(ds, shuffle=False, **loader_options) for ds in test_subsets]

print("Task | digits | train images | test images")
for i, (pair, train_subset, test_subset) in enumerate(zip(TASKS, train_subsets, test_subsets), 1):
    print(f"{i:>4} | {pair[0]}/{pair[1]:<5} | {len(train_subset):>12,} | {len(test_subset):>11,}")


## 4. Model and measurement

The network flattens each image and uses two 1,024-unit ReLU hidden layers before a shared 10-logit output. It has about 1.9 million trainable parameters: substantial for MNIST, but small enough for an 8 GB GPU. We intentionally omit dropout, replay, weight penalties, and task-specific outputs to expose the behavior of plain SGD.

For each task, we train for a fixed number of epochs and continue with the **same weights and optimizer** on the next task. We record mean training loss per epoch and test accuracy for every seen task after each stage. Evaluation uses `model.eval()` and `torch.inference_mode()`; training uses `model.train()` and regular backpropagation.


In [ ]:
class DigitMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28 * 28, 1024),
            nn.ReLU(),
            nn.Linear(1024, 1024),
            nn.ReLU(),
            nn.Linear(1024, 10),
        )

    def forward(self, images):
        return self.network(images)


parameter_count = sum(p.numel() for p in DigitMLP().parameters())
print(f"Trainable parameters: {parameter_count:,}")

EPOCHS_PER_TASK = 4
LEARNING_RATE = 0.05


def train_one_epoch(model, loader, optimizer, loss_fn, device):
    model.train()
    loss_sum = 0.0
    example_count = 0
    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        logits = model(images)
        loss = loss_fn(logits, labels)
        loss.backward()
        optimizer.step()
        loss_sum += loss.item() * labels.size(0)
        example_count += labels.size(0)
    return loss_sum / example_count


@torch.inference_mode()
def accuracy_on_loader(model, loader, device):
    model.eval()
    correct = 0
    total = 0
    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        predictions = model(images).argmax(dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)
    return correct / total


## 5. Run the sequential experiment (CUDA only)

The only updates below are cross-entropy gradients from the current task. We do **not** reset the model between tasks. The matrix entry `[stage, task]` is test accuracy on that task after finishing the stage. Cells for tasks not yet introduced stay `NaN`.

Four epochs per task are a practical starting point. If the last task is still poorly learned, increase `EPOCHS_PER_TASK`; if you want a faster first pass, lower it. Keep these settings fixed when comparing future methods.


In [ ]:
results = None
model = None

if DEVICE is None:
    print("CUDA is unavailable: sequential training was skipped. Run this cell with a CUDA-enabled PyTorch kernel.")
else:
    model = DigitMLP().to(DEVICE)
    optimizer = torch.optim.SGD(model.parameters(), lr=LEARNING_RATE)
    loss_fn = nn.CrossEntropyLoss()
    accuracy_matrix = np.full((len(TASKS), len(TASKS)), np.nan)
    loss_history = []

    for stage, train_loader in enumerate(train_loaders):
        print(f"\nTask {stage + 1}/{len(TASKS)}: digits {TASK_NAMES[stage]}")
        for epoch in range(EPOCHS_PER_TASK):
            mean_loss = train_one_epoch(model, train_loader, optimizer, loss_fn, DEVICE)
            loss_history.append(mean_loss)
            print(f"  Epoch {epoch + 1}/{EPOCHS_PER_TASK} | train loss {mean_loss:.4f}")

        # Evaluate only tasks learned so far, using the official test images.
        for task in range(stage + 1):
            accuracy_matrix[stage, task] = accuracy_on_loader(model, test_loaders[task], DEVICE)
        scores = ", ".join(
            f"{TASK_NAMES[task]}: {accuracy_matrix[stage, task]:.1%}"
            for task in range(stage + 1)
        )
        print(f"  Test accuracy | {scores}")

    results = {"accuracy_matrix": accuracy_matrix, "loss_history": np.array(loss_history)}


## 6. Inspect learning and forgetting

The loss curve shows whether SGD fit each incoming task. The accuracy matrix shows the central continual-learning result: read **down a column** to see what happened to one task as later tasks arrived. A strong diagonal and weak older-task cells indicate recency bias. Each accuracy is measured on a held-out test subset, and the output choice is always unrestricted among ten classes.


In [ ]:
if results is not None:
    fig, ax = plt.subplots(figsize=(10, 3.5))
    epoch_numbers = np.arange(1, len(results["loss_history"]) + 1)
    ax.plot(epoch_numbers, results["loss_history"], marker="o", markersize=3)
    for boundary in range(1, len(TASKS)):
        ax.axvline(boundary * EPOCHS_PER_TASK + 0.5, color="0.65", linestyle="--", linewidth=1)
    ax.set(xlabel="Cumulative epoch", ylabel="Mean training cross-entropy", title="Training loss through the task stream")
    ax.set_xticks(epoch_numbers)
    fig.tight_layout()
    plt.show()
else:
    print("Run the CUDA experiment to generate the loss curve.")


In [ ]:
if results is not None:
    matrix = results["accuracy_matrix"] * 100
    cmap = plt.get_cmap("viridis").copy()
    cmap.set_bad(color="0.9")
    fig, ax = plt.subplots(figsize=(8, 5))
    image = ax.imshow(np.ma.masked_invalid(matrix), vmin=0, vmax=100, cmap=cmap)
    ax.set(xticks=np.arange(len(TASKS)), yticks=np.arange(len(TASKS)),
           xticklabels=TASK_NAMES, yticklabels=TASK_NAMES,
           xlabel="Test task", ylabel="Training completed through task",
           title="Held-out accuracy after each task (%)")
    for row in range(len(TASKS)):
        for col in range(row + 1):
            value = matrix[row, col]
            ax.text(col, row, f"{value:.1f}", ha="center", va="center",
                    color="white" if value < 50 else "black", fontsize=9)
    fig.colorbar(image, ax=ax, label="Accuracy (%)")
    fig.tight_layout()
    plt.show()
else:
    print("Run the CUDA experiment to generate the accuracy matrix.")


A few summaries make the matrix easier to read. **Current task** is the newest pair's accuracy. **Earlier tasks** averages only the previously learned pairs, so it is undefined after task 1. **Seen-task mean** gives each learned task equal weight. These are descriptive diagnostics, not a substitute for the individual task curves.


In [ ]:
if results is not None:
    matrix = results["accuracy_matrix"]
    stages = np.arange(1, len(TASKS) + 1)
    current = np.diag(matrix)
    earlier = np.array([np.nan if stage == 0 else matrix[stage, :stage].mean()
                        for stage in range(len(TASKS))])
    seen_mean = np.array([matrix[stage, :stage + 1].mean()
                          for stage in range(len(TASKS))])

    fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=True)
    for task, name in enumerate(TASK_NAMES):
        axes[0].plot(stages[task:], matrix[task:, task] * 100, marker="o", label=name)
    axes[0].set(title="Each task after it is learned", xlabel="Tasks completed", ylabel="Test accuracy (%)")
    axes[0].legend(title="Digit pair", ncol=2)
    axes[1].plot(stages, current * 100, marker="o", label="Current task")
    axes[1].plot(stages, earlier * 100, marker="o", label="Earlier tasks")
    axes[1].plot(stages, seen_mean * 100, marker="o", label="Seen-task mean")
    axes[1].set(title="Current performance versus retention", xlabel="Tasks completed")
    axes[1].legend()
    for ax in axes:
        ax.set(xticks=stages, ylim=(0, 105))
        ax.axhline(10, color="0.7", linestyle=":", linewidth=1)
        ax.grid(alpha=0.2)
    fig.tight_layout()
    plt.show()
else:
    print("Run the CUDA experiment to generate the task trajectories.")


## 7. Quantify the final state

For each task, compare its accuracy **immediately after learning it** with its accuracy after all five tasks. The difference is the observed drop in percentage points. The newest task has a drop of zero by construction. We report average drop over the first four tasks and final mean accuracy over all five. A negative drop would mean a later task helped an earlier one.


In [ ]:
if results is not None:
    matrix = results["accuracy_matrix"]
    immediate = np.diag(matrix)
    final = matrix[-1]
    drops = (immediate - final) * 100
    x = np.arange(len(TASKS))
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.bar(x - 0.2, immediate * 100, width=0.4, label="Just learned")
    ax.bar(x + 0.2, final * 100, width=0.4, label="After all tasks")
    ax.set(xticks=x, xticklabels=TASK_NAMES, ylim=(0, 105),
           xlabel="Digit pair", ylabel="Test accuracy (%)", title="Initial mastery versus final retention")
    ax.legend()
    fig.tight_layout()
    plt.show()

    print(f"Final mean accuracy across five tasks: {final.mean():.1%}")
    print(f"Mean accuracy drop across first four tasks: {drops[:-1].mean():.1f} percentage points")
    for name, drop in zip(TASK_NAMES, drops):
        print(f"  Task {name}: {drop:+.1f} percentage points")
else:
    print("Run the CUDA experiment to measure final retention.")


## 8. Where do the old digits go?

The final confusion matrix tests all 10,000 official test images after the last task. Each row is a true digit and each column is the model's prediction. Row percentages reveal whether old digits are being mistaken for recently trained digits. The predicted-class histogram provides a compact view of any overall recency bias.


In [ ]:
@torch.inference_mode()
def confusion_counts(model, loader, device):
    model.eval()
    counts = torch.zeros((10, 10), dtype=torch.int64)
    for images, labels in loader:
        predictions = model(images.to(device, non_blocking=True)).argmax(dim=1).cpu()
        flat_indices = labels * 10 + predictions
        counts += torch.bincount(flat_indices, minlength=100).reshape(10, 10)
    return counts.numpy()


if results is not None:
    full_test_loader = DataLoader(test_dataset, shuffle=False, **loader_options)
    counts = confusion_counts(model, full_test_loader, DEVICE)
    row_percent = counts / counts.sum(axis=1, keepdims=True) * 100
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    image = axes[0].imshow(row_percent, vmin=0, vmax=100, cmap="magma")
    axes[0].set(xlabel="Predicted digit", ylabel="True digit", title="Final confusion matrix (row %)",
                xticks=np.arange(10), yticks=np.arange(10))
    fig.colorbar(image, ax=axes[0], label="Percent of true digit")
    axes[1].bar(np.arange(10), counts.sum(axis=0), color="tab:orange", label="Predicted")
    axes[1].plot(np.arange(10), counts.sum(axis=1), color="black", marker="o", label="Actual")
    axes[1].set(xlabel="Digit", ylabel="Number of test images", title="Final prediction distribution",
                xticks=np.arange(10))
    axes[1].legend()
    fig.tight_layout()
    plt.show()
else:
    print("Run the CUDA experiment to inspect final predictions.")


## What to take away

Read the diagonal of the accuracy matrix first: it tells you whether the model learned each pair when it arrived. Then read down the earlier columns and compare their initial and final bars: that is the direct evidence for or against forgetting in **this run**. Finally, inspect the confusion matrix to see which new digits attract old examples.

If the results are noisy, rerun with several seeds before drawing a firm conclusion. Save the task order, architecture, optimizer, epochs, and evaluation rule when making later comparisons; changing those alongside a continual-learning method would make the comparison hard to interpret.
